# EOIR Immigration Court Database — Quickstart

This notebook lets you query the EOIR (Executive Office for Immigration Review) immigration court database in under 2 minutes.

The database contains **every immigration court proceeding, charge, hearing, application for relief, bond hearing, and appeal since the 1970s** — over 160 million rows across 97 tables.

**Two ways to use this notebook:**
- **Option A:** You already have `eoir.duckdb` on Google Drive (fastest)
- **Option B:** Build the database from the EOIR zip file on Google Drive

Source: [github.com/ian-nason/eoir-database](https://github.com/ian-nason/eoir-database)

## Setup

In [ ]:
# Install DuckDB
!pip install duckdb -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import duckdb

# ============================================================
# Option A: Connect to a pre-built database on Google Drive
# ============================================================
DB_PATH = '/content/drive/MyDrive/eoir.duckdb'
con = duckdb.connect(DB_PATH, read_only=True)
print('Connected to', DB_PATH)

# ============================================================
# Option B: Build the database from the EOIR zip
# Uncomment the lines below and comment out Option A above.
# ============================================================
# !git clone https://github.com/ian-nason/eoir-database.git /content/eoir-database
# !pip install requests tqdm -q
# !cd /content/eoir-database && python build_database.py \
#     --zip "/content/drive/MyDrive/EOIR Case Data.zip" \
#     --output /content/drive/MyDrive/eoir.duckdb
# con = duckdb.connect('/content/drive/MyDrive/eoir.duckdb', read_only=True)

## Explore the schema

The `_metadata` table lists every table in the database with row counts and descriptions.

In [ ]:
con.sql("""
    SELECT table_name, description, row_count, column_count
    FROM _metadata
    WHERE is_lookup = false
    ORDER BY row_count DESC
""").show()

In [ ]:
# Lookup (reference) tables — these decode the codes used in the core tables
con.sql("""
    SELECT table_name, description, row_count
    FROM _metadata
    WHERE is_lookup = true
    ORDER BY table_name
""").show(max_rows=80)

## Basic queries

### Proceedings by year

Each row in the `proceedings` table represents one proceeding before an immigration judge. The `OSC_DATE` is when the Notice to Appear (NTA, formerly Order to Show Cause) was filed — effectively the start date of the case.

In [ ]:
con.sql("""
    SELECT
        EXTRACT(YEAR FROM OSC_DATE) AS year,
        COUNT(*) AS proceedings
    FROM proceedings
    WHERE OSC_DATE IS NOT NULL
      AND EXTRACT(YEAR FROM OSC_DATE) BETWEEN 2000 AND 2025
    GROUP BY 1
    ORDER BY 1
""").show(max_rows=30)

### Top 10 courts by volume

Immigration courts are identified by `BASE_CITY_CODE`. The `lu_base_city` lookup table has the court name and location. These counts are all-time.

In [ ]:
con.sql("""
    SELECT
        bc.BASE_CITY_NAME AS court,
        COUNT(*) AS proceedings
    FROM proceedings p
    JOIN lu_base_city bc ON p.BASE_CITY_CODE = bc.BASE_CITY_CODE
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").show()

### Top 10 nationalities

Nationality is recorded per case in the `cases` table (`NAT` column). The `lu_nationality` lookup provides country names.

In [ ]:
con.sql("""
    SELECT
        n.NAT_COUNTRY_NAME AS country,
        COUNT(*) AS cases
    FROM cases c
    JOIN lu_nationality n ON c.NAT = n.NAT_CODE
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").show()

## Timeline analysis

### Days from NTA to first hearing, by court

The `v_first_hearing` view calculates the number of days between the NTA filing date (`OSC_DATE`) and the first hearing for each case. Long wait times are a defining feature of the immigration court backlog.

This shows median wait times for cases filed since 2020 at the 15 busiest courts.

In [ ]:
con.sql("""
    SELECT
        bc.BASE_CITY_NAME AS court,
        COUNT(*) AS cases,
        MEDIAN(fh.days_osc_to_hearing) AS median_days,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY fh.days_osc_to_hearing) AS p25,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY fh.days_osc_to_hearing) AS p75
    FROM v_first_hearing fh
    JOIN lu_base_city bc ON fh.BASE_CITY_CODE = bc.BASE_CITY_CODE
    WHERE fh.OSC_DATE >= '2020-01-01'
    GROUP BY 1
    HAVING COUNT(*) >= 1000
    ORDER BY 3 DESC
    LIMIT 15
""").show()

### Detained vs. non-detained wait times

Detained respondents (`CUSTODY = 'D'`) are held in ICE custody while their cases proceed. Their hearings are typically much faster than non-detained cases (`CUSTODY = 'N'` or `'R'` for released).

In [ ]:
con.sql("""
    SELECT
        CASE CUSTODY
            WHEN 'D' THEN 'Detained'
            WHEN 'N' THEN 'Never detained'
            WHEN 'R' THEN 'Released'
            ELSE 'Other/Unknown'
        END AS custody_status,
        COUNT(*) AS cases,
        MEDIAN(days_osc_to_hearing) AS median_days_to_hearing,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY days_osc_to_hearing) AS p25,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY days_osc_to_hearing) AS p75
    FROM v_first_hearing
    WHERE OSC_DATE >= '2020-01-01'
    GROUP BY 1
    ORDER BY 3
""").show()

## Representation

### % of cases with counsel

The `representatives` table tracks attorney assignments. `STRATTYTYPE = 'ALIEN'` means the respondent has their own attorney (as opposed to a government attorney or pro bono representative).

Representation rates vary dramatically by court and custody status. Detained individuals are far less likely to have a lawyer.

In [ ]:
con.sql("""
    WITH case_rep AS (
        SELECT
            c.IDNCASE,
            c.CUSTODY,
            MAX(CASE WHEN r.STRATTYTYPE = 'ALIEN' THEN 1 ELSE 0 END) AS has_counsel
        FROM cases c
        LEFT JOIN representatives r ON c.IDNCASE = r.IDNCASE
        WHERE c.INPUT_DATE >= '2020-01-01'
        GROUP BY 1, 2
    )
    SELECT
        CASE CUSTODY
            WHEN 'D' THEN 'Detained'
            WHEN 'N' THEN 'Never detained'
            WHEN 'R' THEN 'Released'
            ELSE 'Other'
        END AS custody_status,
        COUNT(*) AS total_cases,
        SUM(has_counsel) AS with_counsel,
        ROUND(100.0 * SUM(has_counsel) / COUNT(*), 1) AS pct_with_counsel
    FROM case_rep
    GROUP BY 1
    ORDER BY 4
""").show()

## Outcomes

### Decision breakdown for removal cases

The `DEC_CODE` in the `proceedings` table records the judge's decision. The `lu_court_decision` lookup decodes it — but note it's a compound key: the same code means different things for different case types (`CASE_TYPE`). Here we look at removal cases (`RMV`) only.

In [ ]:
con.sql("""
    SELECT
        d.strDecDescription AS decision,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM proceedings p
    JOIN lu_court_decision d
        ON p.DEC_CODE = d.strDecCode AND p.CASE_TYPE = d.strCaseType
    WHERE p.CASE_TYPE = 'RMV'
      AND p.COMP_DATE >= '2020-01-01'
      AND p.DEC_CODE IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 15
""").show()

### Asylum application grant rates

The `applications` table tracks applications for relief filed during proceedings. Application type codes are decoded via `lu_application`. Decision codes come from `lu_app_decision`.

In [ ]:
con.sql("""
    SELECT
        ad.strCourtApplnDecDesc AS decision,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM applications a
    JOIN lu_application la ON a.APPL_CODE = la.strcode
    JOIN lu_app_decision ad ON a.APPL_DEC_CODE = ad.strCourtApplnDecCode
    WHERE LOWER(la.strdescription) LIKE '%asylum%'
      AND a.APPL_RECD_DATE >= '2020-01-01'
    GROUP BY 1
    ORDER BY 2 DESC
""").show()

---

## Next steps

- Explore the full table inventory with `SELECT * FROM _metadata`
- See [examples/query_examples.sql](https://github.com/ian-nason/eoir-database/blob/main/examples/query_examples.sql) for 15+ more queries
- See [examples/nyv_detained_237.py](https://github.com/ian-nason/eoir-database/blob/main/examples/nyv_detained_237.py) for a full analysis script
- File issues or contribute at [github.com/ian-nason/eoir-database](https://github.com/ian-nason/eoir-database)